# 01. Data mart
Loading raw Home Credit tables into DuckDB and building a client-level data mart.

In [1]:
import duckdb
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
DB_PATH = PROJECT_ROOT / "data" / "home_credit.duckdb"

assert RAW_DIR.exists(), f"Raw data folder not found: {RAW_DIR}"

con = duckdb.connect(str(DB_PATH))
print("Connected to", DB_PATH)

Connected to d:\Data Science\Projects\Credit Scoring Project\credit-scoring-home-credit\data\home_credit.duckdb


In [2]:
TABLES = {
    "application_train": "application_train.csv",
    "bureau": "bureau.csv",
    "bureau_balance": "bureau_balance.csv",
    "previous_application": "previous_application.csv",
    "pos_cash_balance": "POS_CASH_balance.csv",
    "credit_card_balance": "credit_card_balance.csv",
    "installments_payments": "installments_payments.csv",
}

for table_name, file_name in TABLES.items():
    file_path = (RAW_DIR / file_name).as_posix()
    con.execute(f"""
        CREATE OR REPLACE TABLE {table_name} AS
        SELECT * FROM read_csv('{file_path}', header = true, sample_size = -1)
    """)
    print(f"Loaded {table_name}")

Loaded application_train
Loaded bureau
Loaded bureau_balance
Loaded previous_application
Loaded pos_cash_balance
Loaded credit_card_balance
Loaded installments_payments


In [6]:
import pandas as pd

desc_df = pd.read_csv(
    RAW_DIR / "HomeCredit_columns_description.csv",
    encoding="cp1252",
    index_col=0,
)

con.register("desc_df", desc_df)
con.execute("CREATE OR REPLACE TABLE columns_description AS SELECT * FROM desc_df")
con.unregister("desc_df")

con.sql("SELECT * FROM columns_description LIMIT 5").show()

┌──────────────────────────────┬────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────┐
│            Table             │        Row         │                                                                                         Description                                                                                         │ Special │
│           varchar            │      varchar       │                                                                                           varchar                                                                                           │ varchar │
├──────────────────────────────┼────────────────────┼─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [8]:
EXPECTED_ROWS = {
    "application_train": 307_511,
    "bureau": 1_716_428,
    "bureau_balance": 27_299_925,
    "previous_application": 1_670_214,
    "pos_cash_balance": 10_001_358,
    "credit_card_balance": 3_840_312,
    "installments_payments": 13_605_401,
}

for table_name, expected in EXPECTED_ROWS.items():
    actual = con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
    status = "OK" if actual == expected else "MISMATCH"
    print(f"{table_name:25s} {actual:>12,} {status}")

application_train              307,511 OK
bureau                       1,716,428 OK
bureau_balance              27,299,925 OK
previous_application         1,670,214 OK
pos_cash_balance            10,001,358 OK
credit_card_balance          3,840,312 OK
installments_payments       13,605,401 OK


In [9]:
con.close()